### O que está feito
* Heurísticas construtivas (VMP e IMB)
* Buscas locais (swap, shift e inversão)
* VND incompleto para as buscas locais
* Algoritmo genérico

### TODO
* Revisar se os código de benchmark tão certos (feito por IA)
* registrar todos os resultados das gerações do memético pra fazer a média
* Rever o elitismo e a mutação: elitismo muito estrito e mutação muito alta
* Ver se as seeds tão funcionando (reprodutibilidade)
* resultados tão muito bons. Suspeito
* Fixar número de iterações - agora estamos com 30, talvez reduzir pra 20 tal como Menezes et al (2025).
* Entender o que fazer com os valores intermediários das gerações - aparentemente dá p analisar a convergência
* Analisar necessidade/viabilidade de algum tuning de parâmetros (Menezes et al usa irace)
### Possíveis perguntas
* Comparação dos ganhos de cada etapa: heurísitcas construtivas (VMP, IMB) e busca local -> genético -> memético. Mostrar taxa de ganho a cada etapa.
* Análise da estratégia de busca local que usamos (**TODO: Pesquisar o nome na literatura!!**)
* Comparar memético e genético - algum teste estatístico
* Influencia dos ganhos iniciais do memético

## Pacotes e Bibliotecas

In [1]:
!pip install openpyxl pandas numpy scipy

In [2]:
import pandas as pd
import numpy as np
import openpyxl
import math
import random
import time
import os

## Grafos utilizados e funções auxiliares

### Grafos

Para conseguir representar as matrizes de adjacência dos grafos, foi utilizado o recurso DataFrames da Biblioteca Pandas.

In [ ]:
matriz_problema_km = pd.read_excel("../PCV__Matriz_do_problema.xlsx", sheet_name="Km")
matriz_problema_min = pd.read_excel("../PCV__Matriz_do_problema.xlsx", sheet_name="Min")
cidades = pd.read_excel("../PCV__Matriz_do_problema.xlsx", sheet_name="Cidades")

### Funções Auxiliares

#### Função Calcular custo


A função `calcular_custo` serve para somar os pesos das arestas percorridas por um caminho no grafo, funcionando como a medida de qualidade das soluções no algoritmo genético ou memético.

Recebe uma lista de vértices que representa o percurso e percorre essa lista par a par, pegando o custo de cada aresta diretamente na matriz de adjacência do grafo. Retorna o custo total.
##### Exemplo:
Em um grafo onde a aresta entre 1 -> 3 custa 2 e 3 -> 2 custa 3, a solução `[1,2,3]` vai gerar um custo final de valor 5.


In [4]:
# Função para obter o custo total a partir de uma solucao (caminho)
def calcular_custo(solucao, grafo):
    if len(solucao) <= 1:
        return
    custo = 0
    for prox in range(1, len(solucao)):
        v1 = solucao[prox - 1]
        v2 = solucao[prox]
        custo += grafo[v1][v2 - 1]
    return custo

In [5]:
def calcular_custo_opt(solucao, grafo_np):
    if len(solucao) <= 1:
        return 0

    # HACK: Guarda pra caso a primeira coluna nao esteja eliminada
    if grafo_np.shape[1] == grafo_np.shape[0] + 1:
        grafo_limpo = grafo_np[:, 1:]
    else:
        grafo_limpo = grafo_np

    custo = 0
    for prox in range(1, len(solucao)):
        v1 = solucao[prox - 1] - 1
        v2 = solucao[prox] - 1
        custo += grafo_limpo[v1, v2]
    return custo

#### Busca Local Primeira Melhoria

In [6]:
# Executa busca local usando a estratégia de primeira melhoria.
# 'heuristica' é a função que gera vizinhos (shift, swap, inversao).
def busca_local_primeira_melhoria(solucao, custo_original, heuristica, grafo_np):
    custo_atual = custo_original
    solucao_atual = solucao
    melhoria = True
    while melhoria:
        melhoria = False

        # Gera soluções vizinhas usando a heurística escolhida (shift, swap etc.)
        for nova_solucao in heuristica(solucao_atual):
            novo_custo = calcular_custo_opt(nova_solucao, grafo_np)


            # Se a vizinha for melhor, atualiza e reinicia a busca
            if novo_custo < custo_atual:
                solucao_atual = nova_solucao
                custo_atual = novo_custo
                melhoria = True
                break

    return solucao_atual, custo_atual

In [7]:
def busca_local_vnd(solucao_inicial, custo_inicial, heuristicas_busca_local, grafo_np):
    melhor_caminho = list(solucao_inicial)
    melhor_custo = custo_inicial

    execucoes_por_heuristica = {h.__name__: 0 for h in heuristicas_busca_local}
    num_reinicios_vnd = -1

    # TODO: Perturbacao
    k = 0
    while k < len(heuristicas_busca_local):
        if k == 0:
            num_reinicios_vnd += 1

        heuristica = heuristicas_busca_local[k]
        execucoes_por_heuristica[heuristica.__name__] += 1

        solucao, custo_local = busca_local_primeira_melhoria(melhor_caminho, melhor_custo, heuristica, grafo_np)

        if custo_local < melhor_custo:
            # print(f"Melhoria no VND. Custo: {custo_local}; BL: {heuristicas_busca_local[k].__name__}")
            melhor_caminho = solucao
            melhor_custo = custo_local
            k = 0  # reinicia o VND na primeira vizinhança
        else:
            k += 1

    return melhor_caminho, melhor_custo, execucoes_por_heuristica, num_reinicios_vnd

#### Heurísticas

In [8]:
# Heurísticas que geram uma vizinhança (usam yield)
# Sempre preservam o primeiro e último vértice do caminho

def shift(solucao: list):
    # Move um vértice para outra posição
    for idx_vert_shift in range(1, len(solucao) - 1):
        for idx_destino_shift in range(2, len(solucao) - 1):
            # Cópia para não alterar a original
            solucao_cp = solucao[:]
            # Remove o vértice da posição atual
            vert_shift = solucao_cp.pop(idx_vert_shift)
            # Insere em outra posição
            solucao_cp.insert(idx_destino_shift, vert_shift)
            # Retorna nova solução
            yield solucao_cp


def swap(solucao: list):
    # Troca dois vértices de posição
    for idx_vert_swap in range(1, len(solucao) - 1):
        for idx_destino_swap in range(2, len(solucao) - 1):
            solucao_cp = solucao[:]
            # Troca simples entre duas posições
            solucao_cp[idx_vert_swap], solucao_cp[idx_destino_swap] = (
                solucao_cp[idx_destino_swap],
                solucao_cp[idx_vert_swap]
            )
            yield solucao_cp


def inversao(solucao: list):
    # Inverte um segmento interno do caminho
    for idx_inicio in range(1, len(solucao) - 1):
        for idx_fim in range(2, len(solucao) - 1):
            # Segmento a inverter
            sublista = solucao[idx_inicio:idx_fim]
            # Reconstrói a solução com o segmento invertido
            yield solucao[:idx_inicio] + sublista[::-1] + solucao[idx_fim:]


## Busca por vizinho mais próximo

In [9]:
# Estamos considerando o label numerico
def vizinho_mais_proximo(grafo : pd.DataFrame, vertice_inicial : int | str):
    # Criando conjunto de vertices nao visitados
    # o conjunto contem os indices referentes ao vertice na matriz problema
    vertices_nao_visitados = set(grafo.index)
    percurso = [vertice_inicial]
    vertices_nao_visitados.remove(vertice_inicial - 1)
    dist_total = 0
    v_atual = vertice_inicial
    while len(vertices_nao_visitados) > 0:
        # Obtem o label (numero) do vertice com menor custo
        idx_vizinho_mais_proximo = grafo[v_atual].loc[list(vertices_nao_visitados)].idxmin(skipna=True)
        prox = idx_vizinho_mais_proximo + 1
        dist_total += grafo[v_atual][idx_vizinho_mais_proximo]
        percurso.append(int(prox))
        v_atual = prox
        vertices_nao_visitados.remove(idx_vizinho_mais_proximo)
    dist_total += grafo[v_atual][vertice_inicial -1]
    percurso.append(vertice_inicial)
    return percurso, dist_total

resultados_vmp = [None] * 12
resultados_vmp_busca_local = [None] * 12

## Inserção mais barata

In [10]:
# Estamos considerando o label numerico
def insercao_mais_barata(grafo : pd.DataFrame, vertice_inicial : int | str):

    vertices_restantes = set(grafo.index)

    # Selecionamos o indice do vertice mais proximo a raiz de forma gulosa
    v_mais_proximo = int(grafo[vertice_inicial].idxmin())

    # Ciclo inicial
    rota = [vertice_inicial, v_mais_proximo + 1, vertice_inicial]
    vertices_restantes.remove( vertice_inicial - 1, )
    vertices_restantes.remove(v_mais_proximo)
    vertices_inseridos = {vertice_inicial - 1, v_mais_proximo}

    custo_total = 0
    while vertices_restantes:
        # Estamos selecionando o vertice de 'vertices_restantes' que tem a menor distancia a outro vertice
        # Filtra apenas as linhas da matriz referentes aos vertices do conjunto de vertices_restantes
        r = int(grafo[list(vertices_restantes)].iloc[list(vertices_inseridos)].min().idxmin())

        # Consegue o indice do vertice do conjunto 'vertices_restantes' que tem a menor distancia
        # ao algum vertice incluido na rota. Para manter este codigo similar ao pseudocodigo visto em sala, chamaremos a variavel de
        # Encontrar a melhor posicao para inserir a rota
        melhor_custo_insercao = math.inf
        melhor_posicao_insercao = None
        for i in range(1, len(rota) - 1):
            u, v = rota[i] - 1, rota[i + 1] - 1
            custo = grafo[u + 1][r] + grafo[r + 1][v] - grafo[u + 1][v]
            if custo < melhor_custo_insercao:
                melhor_custo_insercao = custo
                melhor_posicao_insercao = i

        rota.insert(melhor_posicao_insercao + 1, r + 1)
        vertices_restantes.remove(r)
        vertices_inseridos.add(r)
        custo_total += melhor_custo_insercao
    return rota, calcular_custo(rota, grafo)

resultados_imb = [None] * 12
resultados_imb_busca_local = [None] * 12


In [11]:
def aplicar_problema(grafo, inicial, heuristica_construtiva, heuristicas_busca_local, out=True):
    # Solução inicial via heurística construtiva
    caminho, custo = heuristica_construtiva(grafo, inicial)
    melhor_caminho, melhor_custo = caminho, custo
    heuristica_melhor_solucao = None

    if out:
        print(f"Aplicando Heurística Construtiva: {heuristica_construtiva.__name__}")
        print(f"Caminho inicial: {caminho}")
        print(f"Custo inicial: {custo}")
        print("\nAplicando Busca Local:")

    # Aplicação das heurísticas de busca local
    for heuristica in heuristicas_busca_local:
        solucao, custo_local = busca_local_primeira_melhoria(
            melhor_caminho, melhor_custo, heuristica, grafo
        )

        if out:
            print(f"\nHeurística: {heuristica.__name__}")
            print("Método: Primeira Melhoria")
            print(f"Caminho obtido: {solucao}")
            print(f"Custo obtido: {custo_local}")

        # Atualiza melhor solução
        if custo_local < melhor_custo:
            melhor_caminho, melhor_custo = solucao, custo_local
            heuristica_melhor_solucao = heuristica.__name__

    return melhor_caminho, melhor_custo, heuristica_melhor_solucao


In [12]:
def aplicar_problema_rastreado(grafo, inicial, heuristica_construtiva, heuristicas_busca_local, grafo_np, out=False):
    # Gera a solução inicial via Heurística Construtiva
    caminho, custo = heuristica_construtiva(grafo, inicial)

    caminho_atual = list(caminho)
    custo_atual = custo

    execucoes_por_heuristica = {h.__name__: 0 for h in heuristicas_busca_local}

    # Pipeline sequencial de Busca Local
    for heuristica in heuristicas_busca_local:
        # Contador de execuções (iterações da busca local para esta vizinhança)
        melhoria = True
        while melhoria:
            melhoria = False
            execucoes_por_heuristica[heuristica.__name__] += 1
            for nova_solucao in heuristica(caminho_atual):
                novo_custo = calcular_custo_opt(nova_solucao, grafo_np)
                if novo_custo < custo_atual:
                    caminho_atual = nova_solucao
                    custo_atual = novo_custo
                    melhoria = True
                    break

    return caminho_atual, custo_atual, caminho, custo, execucoes_por_heuristica

In [13]:
def aplicar_problema_vnd(grafo, inicial, heuristica_construtiva, heuristicas_busca_local, out=True):
    caminho, custo = heuristica_construtiva(grafo, inicial)
    caminho_inicial = list(caminho)
    custo_inicial = custo
    grafo_np = grafo.to_numpy()

    if out:
        print(f"\nAplicando Heurística Construtiva: {heuristica_construtiva.__name__}")
        print(f"Caminho inicial: {caminho}")
        print(f"Custo inicial: {custo}")
        print("\nAplicando Busca Local com VND (Modo Encapsulado):")

    melhor_caminho, melhor_custo, execucoes_por_heuristica, num_reinicios_vnd = busca_local_vnd(
        caminho_inicial, custo_inicial, heuristicas_busca_local, grafo_np
    )

    if out:
        print(f"Busca Local Concluída!")
        print(f"Custo Final: {melhor_custo}")

    return melhor_caminho, melhor_custo, caminho_inicial, custo_inicial, execucoes_por_heuristica, num_reinicios_vnd

In [14]:
# Inicialização das submatrizes a partir dos DataFrames originais
# 36 Cidades
matriz_problema_km_36 = matriz_problema_km[list(range(1, 37))].iloc[0:36].copy()
matriz_problema_min_36 = matriz_problema_min[list(range(1, 37))].iloc[0:36].copy()

# 24 Cidades
matriz_problema_km_24 = matriz_problema_km[list(range(1, 25))].iloc[0:24].copy()
matriz_problema_min_24 = matriz_problema_min[list(range(1, 25))].iloc[0:24].copy()

# 12 Cidades
matriz_problema_km_12 = matriz_problema_km[list(range(1, 13))].iloc[0:12].copy()
matriz_problema_min_12 = matriz_problema_min[list(range(1, 13))].iloc[0:12].copy()

# 7 Cidades
cidades_7 = [1, 7, 8, 9, 10, 11, 12]
indices_7 = [c - 1 for c in cidades_7]
matriz_problema_km_7 = matriz_problema_km[cidades_7].iloc[indices_7].copy()
matriz_problema_km_7.index = range(len(cidades_7))
matriz_problema_km_7.columns = range(1, len(cidades_7) + 1)

matriz_problema_min_7 = matriz_problema_min[cidades_7].iloc[indices_7].copy()
matriz_problema_min_7.index = range(len(cidades_7))
matriz_problema_min_7.columns = range(1, len(cidades_7) + 1)

# 6 Cidades
matriz_problema_km_6 = matriz_problema_km[list(range(1, 7))].iloc[0:6].copy()
matriz_problema_min_6 = matriz_problema_min[list(range(1, 7))].iloc[0:6].copy()

# Definição das instâncias mapeadas
instancias_problema = [
    ('km_48', matriz_problema_km, 0),
    ('min_48', matriz_problema_min, 1),
    ('km_36', matriz_problema_km_36, 2),
    ('min_36', matriz_problema_min_36, 3),
    ('km_24', matriz_problema_km_24, 4),
    ('min_24', matriz_problema_min_24, 5),
    ('km_12', matriz_problema_km_12, 6),
    ('min_12', matriz_problema_min_12, 7),
    ('km_7', matriz_problema_km_7, 8),
    ('min_7', matriz_problema_min_7, 9),
    ('km_6', matriz_problema_km_6, 10),
    ('min_6', matriz_problema_min_6, 11)
]

print(f'Total de instâncias mapeadas: {len(instancias_problema)}')

Total de instâncias mapeadas: 12


In [15]:
metricas_guloso = []

# TODO: fixar a instanciação em uma celula
problemas_mapeados = [
    ('problema_1_km', matriz_problema_km, 0),
    ('problema_2_min', matriz_problema_min, 1),
    ('problema_3_km', matriz_problema_km_36, 2),
    ('problema_4_min', matriz_problema_min_36, 3),
    ('problema_5_km', matriz_problema_km_24, 4),
    ('problema_6_min', matriz_problema_min_24, 5),
    ('problema_7_km', matriz_problema_km_12, 6),
    ('problema_8_min', matriz_problema_min_12, 7),
    ('problema_9_km', matriz_problema_km_7, 8),
    ('problema_10_min', matriz_problema_min_7, 9),
    ('problema_11_km', matriz_problema_km_6, 10),
    ('problema_12_min', matriz_problema_min_6, 11)
]

# buscas locais a serem executadas sequencialmente.
# WARN: A ordem aqui determina qual busca é exaurida primeiro. Fundamentar!!
# TODO : Entender onde essa abordagem entra na literatura. Aparentemente é alguma
# variação do VND (https://link.springer.com/rwe/10.1007/978-3-319-07124-4_9)
heuristicas_bl = [swap, shift, inversao]

for nome, grafo, idx in problemas_mapeados:
    # VMP + BLs
    t0 = time.perf_counter()
    rota_final_vmp, custo_final_vmp, rota_init_vmp, custo_init_vmp, contagens_vmp = aplicar_problema_rastreado(
        grafo, 1, vizinho_mais_proximo, heuristicas_bl, grafo.to_numpy()
    )
    tempo_total_vmp = time.perf_counter() - t0


    resultados_vmp[idx] = rota_init_vmp
    resultados_vmp_busca_local[idx] = rota_final_vmp

    metricas_guloso.append({
        'Problema': nome,
        'Heuristica Construtiva': 'VMP',
        'Custo Inicial': custo_init_vmp,
        'Melhor Custo Final': custo_final_vmp,
        'Tempo de Execução (s)': tempo_total_vmp,
        'Execuções Swap': contagens_vmp['swap'],
        'Execuções Shift': contagens_vmp['shift'],
        'Execuções Inversao': contagens_vmp['inversao'],
        'Melhor Rota': rota_final_vmp
    })

    # IMB + BLs
    t0 = time.perf_counter()
    rota_final_imb, custo_final_imb, rota_init_imb, custo_init_imb, contagens_imb = aplicar_problema_rastreado(
        grafo, 1, insercao_mais_barata, heuristicas_bl, grafo.to_numpy()
    )
    tempo_total_imb = time.perf_counter() - t0

    # Registra no array global
    resultados_imb[idx] = rota_init_imb
    resultados_imb_busca_local[idx] = rota_final_imb

    metricas_guloso.append({
        'Problema': nome,
        'Heuristica Construtiva': 'IMB',
        'Custo Inicial': custo_init_imb,
        'Melhor Custo Final': custo_final_imb,
        'Tempo de Execução (s)': tempo_total_imb,
        'Execuções Swap': contagens_imb['swap'],
        'Execuções Shift': contagens_imb['shift'],
        'Execuções Inversao': contagens_imb['inversao'],
        'Melhor Rota': rota_final_imb
    })

# TODO: IA que fez. Melhorar
df_detalhado_pipeline = pd.DataFrame(metricas_guloso)
display(df_detalhado_pipeline.head(10))

,Problema,Heuristica Construtiva,Custo Inicial,Melhor Custo Final,Tempo de Execução (s),Execuções Swap,Execuções Shift,Execuções Inversao,Melhor Rota
0,problema_1_km,VMP,2291.4,2096.0,0.764567,8,13,3,"[1, 23, 4, 3, 2, 11, 10, 12, 9, 8, 21, 22, 48,..."
1,problema_1_km,IMB,2148.1,2046.2,0.232273,5,4,4,"[1, 10, 12, 9, 8, 21, 22, 48, 40, 36, 25, 31, ..."
2,problema_2_min,VMP,2355.0,2127.0,0.352362,7,6,9,"[1, 3, 4, 23, 41, 30, 6, 27, 44, 28, 15, 18, 4..."
3,problema_2_min,IMB,2197.0,2107.0,0.306738,4,7,3,"[1, 8, 9, 12, 10, 38, 20, 17, 25, 40, 48, 22, ..."
4,problema_3_km,VMP,1951.5,1716.9,0.184403,11,4,8,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 21, 22, 36,..."
5,problema_3_km,IMB,1835.3,1743.2,0.102105,2,3,5,"[1, 10, 12, 9, 8, 21, 22, 36, 25, 31, 17, 7, 1..."
6,problema_4_min,VMP,1950.0,1728.0,0.251300,9,3,8,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 8, 21, 30, ..."
7,problema_4_min,IMB,1931.0,1765.0,0.240774,1,8,5,"[1, 8, 9, 12, 10, 20, 17, 25, 22, 36, 31, 7, 1..."
8,problema_5_km,VMP,1379.3,1328.6,0.034017,2,3,2,"[1, 23, 2, 11, 10, 12, 9, 8, 21, 22, 17, 7, 19..."
9,problema_5_km,IMB,1459.3,1390.5,0.039033,5,2,1,"[1, 10, 12, 9, 8, 21, 22, 17, 7, 19, 16, 20, 1..."


In [16]:
from datetime import datetime
# TODO: IA que fez. Melhorar
idx_melhores_pipeline = df_detalhado_pipeline.groupby('Problema')['Melhor Custo Final'].idxmin()
df_placar_atualizado_pipeline = df_detalhado_pipeline.loc[idx_melhores_pipeline].copy()

df_placar_atualizado_pipeline['Num_Problema'] = df_placar_atualizado_pipeline['Problema'].str.extract(r'(\d+)').astype(int)
df_placar_atualizado_pipeline = df_placar_atualizado_pipeline.sort_values(by='Num_Problema').drop(columns=['Num_Problema']).reset_index(drop=True)

df_placar_atualizado_pipeline.to_csv(f"resultados_gulosos{datetime.now().strftime("%Y-%m-%d %H:%M")}.csv")

display(df_placar_atualizado_pipeline)

,Problema,Heuristica Construtiva,Custo Inicial,Melhor Custo Final,Tempo de Execução (s),Execuções Swap,Execuções Shift,Execuções Inversao,Melhor Rota
0,problema_1_km,IMB,2148.1,2046.2,0.232273,5,4,4,"[1, 10, 12, 9, 8, 21, 22, 48, 40, 36, 25, 31, ..."
1,problema_2_min,IMB,2197.0,2107.0,0.306738,4,7,3,"[1, 8, 9, 12, 10, 38, 20, 17, 25, 40, 48, 22, ..."
2,problema_3_km,VMP,1951.5,1716.9,0.184403,11,4,8,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 21, 22, 36,..."
3,problema_4_min,VMP,1950.0,1728.0,0.251300,9,3,8,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 8, 21, 30, ..."
4,problema_5_km,VMP,1379.3,1328.6,0.034017,2,3,2,"[1, 23, 2, 11, 10, 12, 9, 8, 21, 22, 17, 7, 19..."
5,problema_6_min,VMP,1283.0,1235.0,0.025619,2,3,2,"[1, 23, 2, 5, 11, 10, 12, 9, 8, 21, 22, 17, 7,..."
6,problema_7_km,IMB,711.0,699.2,0.012884,1,1,2,"[1, 8, 9, 11, 10, 12, 7, 5, 2, 3, 4, 6, 1]"
7,problema_8_min,VMP,687.0,609.0,0.005873,4,3,1,"[1, 8, 9, 7, 12, 10, 11, 5, 2, 6, 4, 3, 1]"
8,problema_9_km,VMP,518.5,438.3,0.002612,9,1,1,"[1, 3, 4, 2, 7, 5, 6, 1]"
9,problema_10_min,VMP,413.0,364.0,0.002431,4,1,1,"[1, 3, 4, 2, 7, 5, 6, 1]"


In [17]:
metricas_guloso_vnd = []
# buscas locais a serem executadas sequencialmente.
# WARN: A ordem aqui determina qual busca é exaurida primeiro. Fundamentar!!
# TODO : Entender onde essa abordagem entra na literatura. Aparentemente é alguma
# variação do VND (https://link.springer.com/rwe/10.1007/978-3-319-07124-4_9)
heuristicas_bl = [swap, shift, inversao]

for nome, grafo, idx in problemas_mapeados:
    # VMP + BLs
    t0 = time.perf_counter()
    rota_final_vmp, custo_final_vmp, rota_init_vmp, custo_init_vmp, contagens_vmp, reinicios_vmp = aplicar_problema_vnd(
        grafo, 1, vizinho_mais_proximo, heuristicas_bl, out=False
    )
    tempo_total_vmp = time.perf_counter() - t0


    resultados_vmp[idx] = rota_init_vmp
    resultados_vmp_busca_local[idx] = rota_final_vmp

    metricas_guloso_vnd.append({
        'Problema': nome,
        'Heuristica Construtiva': 'VMP',
        'Custo Inicial': custo_init_vmp,
        'Melhor Custo Final': custo_final_vmp,
        'Tempo de Execução (s)': tempo_total_vmp,
        'Execuções Swap': contagens_vmp['swap'],
        'Execuções Shift': contagens_vmp['shift'],
        'Execuções Inversao': contagens_vmp['inversao'],
        'Num. Reinícios' : reinicios_vmp,
        'Melhor Rota': rota_final_vmp
    })

    # IMB + BLs
    t0 = time.perf_counter()
    rota_final_imb, custo_final_imb, rota_init_imb, custo_init_imb, contagens_imb, reinicios_imb = aplicar_problema_vnd(
        grafo, 1, insercao_mais_barata, heuristicas_bl, out=False
    )
    tempo_total_imb = time.perf_counter() - t0

    # Registra no array global
    resultados_imb[idx] = rota_init_imb
    resultados_imb_busca_local[idx] = rota_final_imb

    metricas_guloso_vnd.append({
        'Problema': nome,
        'Heuristica Construtiva': 'IMB',
        'Custo Inicial': custo_init_imb,
        'Melhor Custo Final': custo_final_imb,
        'Tempo de Execução (s)': tempo_total_imb,
        'Execuções Swap': contagens_imb['swap'],
        'Execuções Shift': contagens_imb['shift'],
        'Execuções Inversao': contagens_imb['inversao'],
        'Num. Reinícios' : reinicios_imb,
        'Melhor Rota': rota_final_imb
    })

# TODO: IA que fez. Melhorar
df_detalhado_pipeline_vnd = pd.DataFrame(metricas_guloso_vnd)
display(df_detalhado_pipeline_vnd.head(10))

,Problema,Heuristica Construtiva,Custo Inicial,Melhor Custo Final,Tempo de Execução (s),Execuções Swap,Execuções Shift,Execuções Inversao,Num. Reinícios,Melhor Rota
0,problema_1_km,VMP,2291.4,2083.8,0.915827,5,4,2,4,"[1, 23, 4, 3, 2, 11, 10, 12, 9, 8, 21, 22, 48,..."
1,problema_1_km,IMB,2148.1,2046.2,1.075358,4,3,2,3,"[1, 10, 12, 9, 8, 21, 22, 48, 40, 36, 25, 31, ..."
2,problema_2_min,VMP,2355.0,2127.0,1.351505,4,3,2,3,"[1, 3, 4, 23, 41, 30, 6, 27, 44, 28, 15, 18, 4..."
3,problema_2_min,IMB,2197.0,2107.0,1.300634,4,3,2,3,"[1, 8, 9, 12, 10, 38, 20, 17, 25, 40, 48, 22, ..."
4,problema_3_km,VMP,1951.5,1695.0,0.764276,5,4,2,4,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 22, 36, 25, 31..."
5,problema_3_km,IMB,1835.3,1713.9,0.845237,5,4,2,4,"[1, 10, 12, 9, 8, 21, 22, 36, 25, 31, 17, 7, 1..."
6,problema_4_min,VMP,1950.0,1728.0,1.055005,4,3,2,3,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 8, 21, 30, ..."
7,problema_4_min,IMB,1931.0,1765.0,0.972537,3,3,2,2,"[1, 8, 9, 12, 10, 20, 17, 25, 22, 36, 31, 7, 1..."
8,problema_5_km,VMP,1379.3,1321.0,0.395004,5,4,2,4,"[1, 2, 11, 10, 12, 9, 8, 21, 22, 17, 7, 19, 16..."
9,problema_5_km,IMB,1459.3,1390.5,0.239293,3,2,1,2,"[1, 10, 12, 9, 8, 21, 22, 17, 7, 19, 16, 20, 1..."


In [18]:
# TODO: IA que fez. Melhorar
idx_melhores_pipeline_vnd = df_detalhado_pipeline_vnd.groupby('Problema')['Melhor Custo Final'].idxmin()
df_placar_atualizado_pipeline_vnd = df_detalhado_pipeline_vnd.loc[idx_melhores_pipeline].copy()

df_placar_atualizado_pipeline_vnd['Num_Problema'] = df_placar_atualizado_pipeline_vnd['Problema'].str.extract(r'(\d+)').astype(int)
df_placar_atualizado_pipeline_vnd = df_placar_atualizado_pipeline_vnd.sort_values(by='Num_Problema').drop(columns=['Num_Problema']).reset_index(drop=True)

df_placar_atualizado_pipeline_vnd.to_csv(f"resultados_gulosos_vnd_{datetime.now().strftime("%Y-%m-%d %H:%M")}.csv")

display(df_placar_atualizado_pipeline_vnd)

,Problema,Heuristica Construtiva,Custo Inicial,Melhor Custo Final,Tempo de Execução (s),Execuções Swap,Execuções Shift,Execuções Inversao,Num. Reinícios,Melhor Rota
0,problema_1_km,IMB,2148.1,2046.2,1.075358,4,3,2,3,"[1, 10, 12, 9, 8, 21, 22, 48, 40, 36, 25, 31, ..."
1,problema_2_min,IMB,2197.0,2107.0,1.300634,4,3,2,3,"[1, 8, 9, 12, 10, 38, 20, 17, 25, 40, 48, 22, ..."
2,problema_3_km,VMP,1951.5,1695.0,0.764276,5,4,2,4,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 22, 36, 25, 31..."
3,problema_4_min,VMP,1950.0,1728.0,1.055005,4,3,2,3,"[1, 23, 4, 3, 2, 5, 11, 10, 12, 9, 8, 21, 30, ..."
4,problema_5_km,VMP,1379.3,1321.0,0.395004,5,4,2,4,"[1, 2, 11, 10, 12, 9, 8, 21, 22, 17, 7, 19, 16..."
5,problema_6_min,VMP,1283.0,1228.0,0.283514,5,4,2,4,"[1, 2, 5, 11, 10, 12, 9, 8, 21, 22, 17, 7, 19,..."
6,problema_7_km,IMB,711.0,672.7,0.052366,3,3,2,2,"[1, 8, 9, 10, 12, 7, 5, 11, 2, 3, 4, 6, 1]"
7,problema_8_min,VMP,687.0,609.0,0.028480,3,2,1,2,"[1, 8, 9, 7, 12, 10, 11, 5, 2, 6, 4, 3, 1]"
8,problema_9_km,VMP,518.5,438.3,0.005637,2,1,1,1,"[1, 3, 4, 2, 7, 5, 6, 1]"
9,problema_10_min,VMP,413.0,364.0,0.004887,2,1,1,1,"[1, 3, 4, 2, 7, 5, 6, 1]"


## Genético

Um algoritmo genético é uma técnica de otimização inspirada na evolução natural, usada para encontrar boas soluções para problemas complexos, como roteamento, planejamento ou design de sistemas. Ele simula o processo de seleção natural, onde as soluções mais “aptas” têm mais chance de sobreviver e gerar descendentes melhores.

O funcionamento do algoritmo genético envolve algumas etapas principais:

* **População inicial:** Cria-se um conjunto de soluções possíveis (chamadas de indivíduos ou cromossomos). Essas soluções podem ser geradas aleatoriamente ou usando algum critério heurístico.

* **Avaliação:** Cada indivíduo é avaliado de acordo com uma função de custo ou qualidade, que mede quão boa é a solução.

* **Seleção:** Escolhem-se os melhores indivíduos para se tornarem “pais” da próxima geração. Técnicas comuns incluem torneio, roleta e elitismo.

* **Cruzamento:** Os pais combinam partes de suas soluções para gerar descendentes, tentando preservar características boas de cada um.

* **Mutação:** Pequenas alterações são aplicadas nos descendentes para explorar novas soluções e evitar que o algoritmo fique preso em mínimos locais.

* **Renovação:** Determina quais indivíduos entram na próxima geração, mantendo o tamanho da população constante e garantindo que boas soluções não se percam.

O processo se repete por várias gerações até que se alcance uma solução satisfatória ou que o número máximo de iterações seja atingido.

### Gerador de soluções aleatórias

A função `solucao_aleatoria` cria uma rota válida para o problema de roteamento, garantindo que todos os vértices do grafo sejam visitados exatamente uma vez antes de retornar ao ponto inicial. Primeiro, ela cria uma lista com todos os vértices, representados pelos índices do DataFrame, e remove o vértice inicial, que será fixo no início e no fim da rota. Em seguida, embaralha os vértices restantes de forma aleatória, gerando uma permutação diferente a cada chamada. Por fim, concatena o vértice inicial no começo e no fim.

A função `calcular_custo` percorre a rota gerada e soma os custos das arestas entre cada par de vértices consecutivos. Cada valor na matriz do grafo representa o custo de ir de um vértice a outro, e a soma total desses custos fornece o custo completo da solução.


In [19]:
import random
def solucao_aleatoria(grafo : pd.DataFrame, vertice_inicial):
    rota = list(map(lambda i : i + 1,grafo.index))
    rota.remove(vertice_inicial)
    random.shuffle(rota)
    return [vertice_inicial] + rota + [vertice_inicial]


samples = [solucao_aleatoria(matriz_problema_km, 1) for i in range(0,6)]
print(samples)

[[1, 34, 43, 37, 15, 11, 40, 19, 18, 31, 46, 25, 12, 48, 33, 7, 38, 8, 22, 42, 35, 14, 39, 28, 4, 45, 26, 16, 17, 27, 30, 10, 2, 44, 32, 3, 21, 20, 13, 23, 41, 47, 6, 36, 29, 9, 24, 5, 1], [1, 2, 20, 40, 39, 36, 48, 26, 42, 23, 32, 18, 47, 25, 37, 14, 3, 22, 44, 13, 17, 33, 29, 6, 46, 12, 35, 4, 34, 31, 15, 38, 8, 30, 24, 27, 5, 28, 10, 9, 43, 21, 11, 19, 7, 41, 16, 45, 1], [1, 14, 20, 2, 48, 17, 43, 15, 26, 5, 11, 34, 33, 37, 45, 38, 22, 21, 16, 30, 24, 9, 8, 46, 19, 27, 35, 40, 6, 31, 44, 23, 29, 32, 36, 18, 41, 42, 39, 7, 47, 10, 3, 13, 28, 25, 4, 12, 1], [1, 8, 2, 3, 14, 20, 5, 37, 29, 33, 12, 40, 26, 10, 34, 15, 16, 28, 30, 6, 21, 4, 38, 7, 32, 24, 42, 9, 13, 17, 45, 36, 35, 41, 19, 46, 18, 48, 39, 31, 43, 27, 25, 44, 22, 47, 11, 23, 1], [1, 33, 3, 21, 31, 13, 46, 12, 17, 29, 43, 35, 26, 42, 28, 48, 40, 47, 25, 41, 45, 4, 37, 23, 18, 10, 22, 7, 16, 6, 2, 11, 39, 30, 24, 27, 36, 5, 15, 20, 44, 32, 38, 34, 14, 8, 9, 19, 1], [1, 16, 31, 46, 36, 15, 11, 5, 37, 10, 3, 39, 9, 24, 6, 18,

### Populações


Como explicamos acima o conceito de população,agora iremos gerar as populações com 15 indivíduos

In [20]:
def criar_populacao(grafo, resultados_imb, resultados_vmp, resultados_imb_busca_local, resultados_vmp_busca_local, indice, tamanho=40, vertice_inicial=1):

    # Seleciona as melhores solucoes que ja temos. Estes são a elite da população
    # inicial
    solucao_insercao_mais_barata = resultados_imb[indice]
    solucao_vizinho_mais_proximo = resultados_vmp[indice]
    solucao_insercao_mais_barata_local = resultados_imb_busca_local[indice]
    solucao_vizinho_mais_proximo_local = resultados_vmp_busca_local[indice]


    candidatas = [
        solucao_insercao_mais_barata,
        solucao_vizinho_mais_proximo,
        solucao_insercao_mais_barata_local,
        solucao_vizinho_mais_proximo_local
    ]

    solucoes_aleatorias = [solucao_aleatoria(grafo, vertice_inicial) for _ in range(tamanho - len(candidatas))]
    todas_solucoes = solucoes_aleatorias + candidatas
    # Remove duplicatas
    populacao_unica = []
    vistos = set()
    for sol in todas_solucoes:
        tupla_sol = tuple(sol)
        if tupla_sol not in vistos:
            vistos.add(tupla_sol)
            populacao_unica.append(sol)

    while len(populacao_unica) < tamanho:
        populacao_unica.append(solucao_aleatoria(grafo, vertice_inicial))

    cromossomos = []
    for sol in populacao_unica:
        custo = calcular_custo_opt(sol, grafo.to_numpy())
        custo_valido = custo if custo is not None else float('inf')
        cromossomos.append((sol, custo_valido))

    # Ordena os cromossomos de forma crescente com base no custo (segundo elemento da tupla)
    populacao_ordenada = sorted(cromossomos, key=lambda x: x[1])

    print(f"População problema {indice+1} criada. Tamanho final (sem duplicatas): {len(populacao_ordenada)} | Melhor custo: {populacao_ordenada[0][1]:.2f}")

    return populacao_ordenada

### Funções para o algoritmo Genético

#### Representação dos cromossomos


Transformamos cada população em uma lista de cromossomos com seus custos para que todos os operadores genéticos possam avaliar a qualidade da solução, facilitando seleção, cruzamento e mutação de forma consistente.

#### Seleção dos genitores

Seleciona os melhores cromossomos da população e combina cada um com um genitor aleatório, garantindo que soluções de baixo custo sobrevivam enquanto mantém diversidade genética.

In [21]:
def elitismo(populacao : list, num_pares):
  # Ordena a população pelo custo (menor primeiro)
  populacao_ordenada = sorted(populacao, key=lambda x: x[1])
  genitores = []

  for i in range(0,num_pares):
      # Escolhe o segundo genitor aleatoriamente para manter diversidade
      segundo_genitor = random.choice(range(0, num_pares))
      if segundo_genitor == i:
          #segundo_genitor = 0
          random.choice([j for j in range(num_pares) if j != i])
      genitores.append((populacao_ordenada[i], populacao_ordenada[segundo_genitor]))
  return genitores


Escolhe genitores de forma competitiva em torneios aleatórios, selecionando os melhores dentro de cada subconjunto, equilibrando pressão seletiva e diversidade genética.

In [22]:
def torneio(populacao: list, num_pares: int, tamanho_torneio: int = 3):
  genitores = []

  # Para cada par a ser gerado
  for _ in range(num_pares):
    # Seleciona competidores aleatórios e escolhe o de menor custo
    competidores_1 = random.sample(populacao, k=tamanho_torneio)
    pai1 = min(competidores_1, key=lambda x: x[1])
    competidores_2 = random.sample(populacao, k=tamanho_torneio)
    pai2 = min(competidores_2, key=lambda x: x[1])

    # Adiciona o par à lista de genitores
    genitores.append((pai1, pai2))
  return genitores

#### Operadores de crossover

In [23]:
def crossover_1_ponto(cromossomo_1, cromossomo_2):
    n = len(cromossomo_1) # Cromossomos tem comprimentos iguais
    # Ponto de corte
    mid = n // 2
    offspring_1 = cromossomo_1[0:mid] + cromossomo_2[mid:n]
    offspring_2 = cromossomo_2[0:mid] + cromossomo_1[mid:n]
    offsprings = [offspring_1, offspring_2]
    # As labels sao numeros em sequencia
    vertices = set(range(2, n))
    for i, offspring in enumerate(offsprings):
        # vertices ja presentes no offspring atual
        vertices_no_offspring = set(offspring)
        vertices_no_offspring.remove(1)
        # se o numero de elementos unicos eh menor que o tamanho - 1 (ja que eh um ciclo),
        # ja elementos repetidos
        if len(vertices_no_offspring) < n-1:
            vertices_restantes = vertices.difference(vertices_no_offspring)
            novo_offspring = offspring[0:mid]
            controle = set(offspring[1:mid])
            for v in offspring[mid:]:
                novo_v = None
                if v in controle:
                    novo_v = vertices_no_offspring.pop()
                else:
                    novo_v = v
                controle.add(novo_v)
                novo_offspring.append(novo_v)
            offsprings[i] = novo_offspring
    return offsprings[0], offsprings[1]


A função `crossover_ox` gera dois descendentes combinando partes de dois genitores, preservando a ordem e garantindo que cada vértice apareça apenas uma vez.

In [24]:
def crossover_ox(genitor_1, genitor_2):
    # Escolhe um ponto de corte aleatório entre 1 e penúltimo índice
    ponto = random.choice(range(1, len(genitor_1) - 1))

    # Cria os dois descendentes pegando os segmentos iniciais de cada genitor
    offspring_1, offspring_2 = genitor_1[:ponto], genitor_2[:ponto]

    # Conjuntos com os vertices doas duas partições
    vertices_incluidos1, vertices_incluidos2 = set(offspring_1), set(offspring_2)

    # Mesclando os offsprings mantendo as ordens relativas
    for i in genitor_2:
        if i not in vertices_incluidos1:
            offspring_1.append(i)
            vertices_incluidos1.add(i)
    # fechando o ciclo
    offspring_1.append(genitor_1[0])

    for i in genitor_1:
        if i not in vertices_incluidos2:
            offspring_2.append(i)
            vertices_incluidos2.add(i)
    # fechando o ciclo
    offspring_2.append(genitor_2[0])

    return offspring_1, offspring_2

A função `crossover_uniforme` mistura genes dos dois genitores posição a posição, garantindo diversidade genética e mantendo rotas válidas, ideal para introduzir novas combinações sem perder a viabilidade do ciclo

In [25]:
def crossover_uniforme(genitor_1,genitor_2):
    # Inicializa os descendentes como cópias dos genitores
    offspring_1, offspring_2 = genitor_1, genitor_2
    # Conjuntos de genes que ainda podem ser usados (vértices disponíveis)
    genes_faltantes_1, genes_faltantes_2 = (
     set(range(0, len(genitor_1))), set(range(0, len(genitor_1)))
    )

    # Percorre cada posição (exceto início e fim) e decide aleatoriamente de qual genitor o gene virá
    for i in range(1, len(offspring_1) - 1):
        if random.random() < 0.5:
            if offspring_2[i] not in genes_faltantes_1:
                offspring_1[i] = genes_faltantes_1.pop()
            else:
                offspring_1[i] = offspring_2[i]
            genes_faltantes_1.add(offspring_1[i])
        if random.random() < 0.5:
            if offspring_1[i] not in genes_faltantes_2:
                offspring_2[i] = genes_faltantes_2.pop()
            else:
                offspring_2[i] = offspring_1[i]
            genes_faltantes_2.add(offspring_2[i])
    return offspring_1, offspring_2

A função de `cruzamento` aplica o crossover em todos os pares de genitores, cria descendentes e calcula o custo de cada rota. Garante que os novos cromossomos estejam prontos para as próximas etapas do algoritmo genético.

In [26]:
def cruzamento(fn_crossover, genitores, grafo_np : np.ndarray):
  # Lista para armazenar todos os descendentes gerados
    offsprings = []
    for g in genitores:
        genitor_1,genitor_2=g

        # TODO : Parametrizar
        offspring1, offspring2 = fn_crossover(genitor_1[0], genitor_2[0])
        # print(f"Genitores: {genitor_1[0]} e {genitor_2[0]}")
        # print(f"Offsprings: {offspring1[0]} e {offspring2[0]}")
        # Adiciona os descendentes à lista junto com seu custo calculado usando a matriz do grafo
        offsprings.extend([(offspring1, calcular_custo_opt(offspring1, grafo_np)), (offspring2,calcular_custo_opt(offspring2, grafo_np))])
    return offsprings

In [27]:
## Order Crossover (ANTIGO)
# def crossover_ox(genitor_1, genitor_2):
#     offspring_1, offspring_2 = genitor_1[:len(genitor_1) // 2], genitor_2[:len(genitor_2) // 2]
#     vertices_incluidos1, vertices_incluidos2 = set(offspring_1), set(offspring_2)
#     for i in genitor_2:
#         if i not in vertices_incluidos1:
#             offspring_1.append(i)
#     offspring_1.append(genitor_1[0])
#     for i in genitor_1:
#         if i not in vertices_incluidos2:
#             offspring_2.append(i)
#     offspring_2.append(genitor_2[0])
#     return offspring_1, offspring_2
## Partially-Bmapped Crossover (à finalizar)
# https://www.nature.com/articles/s41598-025-86695-4
# def crossover_pbx(genitor_1, genitor_2):
#     posicoes = random.randint(1, len(genitor_1) - 1)
#     pos_genes_1 = set(random.choices(range(1, len(genitor_1) - 1), k=posicoes))
#     pos_genes_2 = set(range(0,48)).difference(genes_1)
#     genes_1 = {genitor_1[i] for i in pos_genes_1}
#     genes_2 = {}
#     offspring_1, offspring_2 = [0 for i in range(48)], [0 for i in range(48)]
#     for i in genes:
#         offspring_1[i] = genitor_1[i]
#     for i in range(0,48):
#         if i in pos_genes_1:
#             continue
#         if genitor_2[i] in genes_1:
#             g = genes_2.pop()
#             offspring_1[i] = g
#     genes_1 = random.choices(range(1, len(genitor_1) - 1), k=posicoes)
#     offspring_1, offspring_2 = genitor_1[:ponto], genitor_2[:ponto]
#     vertices_incluidos1, vertices_incluidos2 = set(offspring_1), set(offspring_2)
#     for i in genitor_2:
#         if i not in vertices_incluidos1:
#             offspring_1.append(i)
#     offspring_1.append(genitor_1[0])
#     for i in genitor_1:
#         if i not in vertices_incluidos2:
#             offspring_2.append(i)
#     offspring_2.append(genitor_2[0])
#     return offspring_1, offspring_2

# def crossover_2_pontos(genitor_1, genitor_2):
#     offspring_1, offspring_2 = genitor_1[:len(genitor_1) // 2], genitor_2[:len(genitor_2) // 2]
#     vertices_incluidos1, vertices_incluidos2 = set(offspring_1), set(offspring_2)
#     g1, g2 = random.sample(range(1,len(genitor_1) - 1), 2)
#     g_1_inicio, g_1_final = min(g1,g2), max(g1,g2)
#     g3, g4 = random.sample(range(1,len(genitor_1) - 1), 2)
#     g_2_inicio, g_2_final = min(g1,g2), max(g1,g2)
#     for i in genitor_2:
#         if i not in vertices_incluidos1:
#             offspring_1.append(i)
#     offspring_1.append(genitor_1[0])
#     for i in genitor_1:
#         if i not in vertices_incluidos2:
#             offspring_2.append(i)
#     offspring_2.append(genitor_2[0])
#     return offspring_1, offspring_2
#     pass

# def crossover_2_pontos(genitor_1, genitor_2):
#     offspring_1, offspring_2 = genitor_1[:len(genitor_1) // 2], genitor_2[:len(genitor_2) // 2]
#     vertices_incluidos1, vertices_incluidos2 = set(offspring_1), set(offspring_2)
#     g1, g2 = random.sample(range(1,len(genitor_1) - 1), 2)
#     g_1_inicio, g_1_final = min(g1,g2), max(g1,g2)
#     g3, g4 = random.sample(range(1,len(genitor_1) - 1), 2)
#     g_2_inicio, g_2_final = min(g1,g2), max(g1,g2)
#     segmento_1 = genitor_1[g1:g2]
#     vertices_segmento_1 = set(segmento_1)
#     vertices_segmento_2 = set(segmento_2)
#     for v in segmento_2:
#         if v in segmento_1:
#
#     segmento_2 = genitor_2[g3:g4]
#
#
#     for i in genitor_2:
#         if i not in vertices_incluidos1:
#             offspring_1.append(i)
#     offspring_1.append(genitor_1[0])
#     for i in genitor_1:
#         if i not in vertices_incluidos2:
#             offspring_2.append(i)
#     offspring_2.append(genitor_2[0])
#     return offspring_1, offspring_2
#     pass

#### Mutação

Troca aleatoriamente dois vértices da rota para introduzir diversidade, evitando que o algoritmo fique preso em mínimos locais. É simples, mas eficaz para explorar novas soluções.

In [28]:
def mutacao_swap(cromossomos, probabilidade, grafo_np : np.ndarray):
    novos_cromossomos = []
    for cromossomo in cromossomos:
        # Extrai o caminho do cromossomo (apenas a lista de vértices)
        cromossomo = list(cromossomo[0])

        # Aplica a mutação com probabilidade definida
        if random.random() < probabilidade:
            # Escolhe dois genes aleatórios e troca suas posições
            gene_1, gene_2 = random.sample(range(1, len(cromossomo) - 1), 2)
            cromossomo[gene_1], cromossomo[gene_2] = cromossomo[gene_2], cromossomo[gene_1]
        # Adiciona o cromossomo (mutado ou não) com seu novo custo
        novos_cromossomos.append((cromossomo, calcular_custo_opt(cromossomo, grafo_np())))

    return novos_cromossomos

Inverte um segmento da rota para criar novas permutações sem alterar demais a solução. Essa mutação é útil para melhorar rotas locais de forma significativa.

In [29]:
def mutacao_inversao(cromossomos, probabilidade, grafo_np):
    novos_cromossomos = []
    for cromossomo_original_tuple in cromossomos:
        # Extrai a rota
        cromossomo = list(cromossomo_original_tuple[0])

        if random.random() < probabilidade:
            # Escolhe dois genes aleatórios e inverte a ordem do segmento entre eles
            gene_1, gene_2 = random.sample(range(1, len(cromossomo) - 1), 2)
            gene_inicio = min(gene_1, gene_2)
            gene_fim = max(gene_1, gene_2)
            parte_invertida = cromossomo[gene_inicio:gene_fim][::-1]
            novo = cromossomo[:gene_inicio] + parte_invertida + cromossomo[gene_fim:]

            novos_cromossomos.append((novo, calcular_custo_opt(novo,grafo_np)))
            # print(f"Custo mutado : {calcular_custo(novo, grafo)}")
        else:
            novos_cromossomos.append(cromossomo_original_tuple)
            # Adiciona o cromossomo (mutado ou não) com seu novo custo

    return novos_cromossomos


#### Renovação da população

A função` renovacao_elitismo` vai manter os melhores cromossomos (menor custo) na população, garantindo que soluções de alta qualidade não sejam perdidas.

In [30]:
def renovacao_elitismo(populacao_atual, offspring, num_populacao):
    # Combina a população atual com os descendentes
    pop = populacao_atual + offspring

    # Ordena todos os cromossomos pelo custo (do menor para o maior)
    pop_ordenada = sorted(pop, key=lambda x: x[1])

    # Mantém apenas os melhores indivíduos para a próxima geração
    return pop_ordenada[:num_populacao]


A funçã `renovacao_torneio` é responsavel pela seleção de forma competitiva: pequenos grupos competem, e o melhor é escolhido. Introduz diversidade, evitando que apenas os melhores de toda a população dominem.

In [31]:
# Renovacao da populacao por torneio
def renovacao_torneio(populacao_atual, offspring, num_populacao):
    # Usando torneio
    nova_populacao = []
    # Combina população atual com descendentes
    pop = populacao_atual + offspring
    while len(nova_populacao) < num_populacao:
        candidatos = random.choices(pop, k=3)
        # offspring_individuo = random.choice(list(offspring))
        # Escolhe o melhor candidato (menor custo)
        escolhido = min(candidatos, key=lambda x: x[1])
        nova_populacao.append(escolhido)
    return nova_populacao


a função `roleta` seleciona cromossomos com probabilidade proporcional à qualidade da solução: melhores soluções têm maior chance de serem escolhidas, mas soluções piores ainda podem participar, mantendo diversidade.

In [32]:
def roleta(populacao, offspring, num_populacao):
    populacao = populacao + offspring
    cromossomos = []

    # Calcula o somatório dos custos
    somatorio_custo = sum(map(lambda cromossomo : cromossomo[1], populacao))

    # Normaliza o custo de cada cromossomo para fitness
    values = [solucao[1]/somatorio_custo for solucao in populacao]
    fitness = []
    for val in values:
        min_val = min(values)
        max_val = max(values)
        normalized_data = (max_val - val) / (max_val - min_val)
        fitness.append(normalized_data)
    print(f"Custos : ", list(map(lambda x : x[1], populacao)))
    print(f"Fitness : ", fitness)
    solucoes = list(map(lambda c : c[1], populacao))

    # Seleciona aleatoriamente cromossomos proporcional ao fitness
    return random.choices(populacao, weights=fitness, k=num_populacao)


#### Função principal

Essa função é o fluxo completo do algoritmo genético: ela recebe a população inicial e itera um número definido de vezes, aplicando seleção, cruzamento e mutação, e renovando a população em cada iteração. Ao final, retorna a população otimizada, onde as melhores soluções (rotas com menor custo) tendem a se destacar.

In [33]:
def algoritmo_genetico(grafo, populacao_inicial, numero_iteracoes, taxa_mutacao, mutacao, selecao, cruzamento, renovacao):
    populacao_atual = populacao_inicial
    grafo_np = grafo.to_numpy()
    for iter in range(0, numero_iteracoes):
        genitores = selecao(populacao_atual, len(populacao_atual))

        # print(f"Numero de genitores: {len(genitores)}")
        offsprings = cruzamento(crossover_ox,genitores, grafo_np)
        # Ja aplica a taxa de mutacao a populacao
        offsprings = mutacao(offsprings, taxa_mutacao, grafo_np)
        # print(f"Tamanho do offspring: {len(offsprings)}")
        # print(offsprings)
        populacao_atual = renovacao(populacao_atual, offsprings, len(populacao_atual))
        # print(f"Tamanho da pop: {len(populacao_atual)}")
    return populacao_atual

### Problemas

In [34]:
def executar_problema(nome_problema, matriz, cromossomos, num_execucoes=20, num_geracoes=100,taxa_mutacao=0.4, mutacao=mutacao_inversao, selecao=elitismo, cruzamento=cruzamento, renovacao=renovacao_elitismo, pasta_resultados="resultados_genetico"):

    # Cria pasta se não existir
    os.makedirs(pasta_resultados, exist_ok=True)

    melhores = []
    tempos = []

    for i in range(num_execucoes):
        inicio = time.perf_counter()
        resultado = algoritmo_genetico(matriz, cromossomos, num_geracoes, taxa_mutacao, mutacao, selecao, cruzamento, renovacao)
        fim = time.perf_counter()
        tempos.append(fim - inicio)
        melhores.append(min(resultado, key=lambda x: x[1]))

    melhores = sorted(melhores, key=lambda x: x[1])


    print(f"Resultados {nome_problema}:")
    print("Melhor solução geral --------")
    print(f"Caminho: {melhores[0][0]}")
    print(f"Custo: {melhores[0][1]}")
    print("---------------------------")
    print("Custo médio: ", sum([custo for _, custo in melhores]) / len(melhores))
    print("Tempo médio: ", sum(tempos) / len(tempos))

    # Salvando em arquivo
    arquivo = os.path.join(pasta_resultados, f"resultado_{nome_problema}.txt")
    with open(arquivo, "w", encoding="utf-8") as f:
        f.write(f"RESULTADOS - {nome_problema}\n")
        f.write(f"CONFIGURAÇÃO:\n")
        f.write(f"  Número de execuções: {num_execucoes}\n")
        f.write(f"  Gerações por execução: {num_geracoes}\n")
        f.write(f"  Taxa de mutação: {taxa_mutacao}\n")
        f.write(f"{'='*60}\nESTATÍSTICAS GERAIS\n{'='*60}\n")
        f.write(f"Melhor caminho encontrado: {melhores[0][0]}\n")
        f.write(f"Melhor custo encontrado: {melhores[0][1]:.2f}\n")
        f.write(f"Custo médio: {(sum([custo for _, custo in melhores])/len(melhores)):.2f}\n")
        f.write(f"Tempo médio: {(sum(tempos)/len(tempos)):.2f}s\n")
        f.write(f"Tempo total: {sum(tempos):.2f}s\n\n")
        f.write(f"{'='*60}\nTODAS SOLUÇÕES ENCONTRADAS\n{'='*60}\n")
        for idx, (rota, custo) in enumerate(melhores):
            f.write(f"\nTop {idx+1} --------\n")
            f.write(f"Custo: {custo:.2f}\n")
            f.write(f"Rota: {rota}\n")
            f.write("---------------------------\n")

    print(f"Resultados salvos em: {arquivo}\n")


## Algoritmo Memetico

In [35]:
def algoritmo_memetico(grafo, populacao_inicial, numero_iteracoes, taxa_mutacao, mutacao, selecao, cruzamento, renovacao, heuristicas_busca_local, probabilidade_busca_local=0.2):
    populacao_atual = populacao_inicial
    melhores = []
    tempos = []

    n_genitores = len(populacao_atual) // 2
    print(f"Numero de genitores: {n_genitores}")

    grafo_np = grafo.to_numpy()
    # Loop principal do algoritmo memético, iterando pelo número de gerações
    for iteracao in range(numero_iteracoes):

        # Registra o tempo de início da iteração
        tempo_começo = time.perf_counter()

        # Seleciona os genitores a partir da população atual usando a função de seleção especificada
        genitores = selecao(populacao_atual, n_genitores)

        offsprings = cruzamento(crossover_ox,genitores, grafo_np)

        # TODO: pensar na taxa de mutacao
        offsprings = mutacao(offsprings, taxa_mutacao, grafo_np)
        novos_offsprings = []

        for item in offsprings:
            # HACK: guarda por causa do problema do cromossomo. tirar dps
            if isinstance(item, tuple):
                individuo = item[0]
            else:
                individuo = item

            custo_individuo = calcular_custo_opt(individuo, grafo_np)

            # NOTE: ver se o RNG ta considerano a seed das celulas. Rodar com
            # menos iterações p/ verificar
            if random.random() < probabilidade_busca_local:
                melhorou = False

                for heuristica in heuristicas_busca_local:
                    nova_rota, novo_custo = busca_local_primeira_melhoria(individuo, custo_individuo, heuristica, grafo_np)
                    if novo_custo < custo_individuo:
                        individuo = nova_rota
                        custo_individuo = novo_custo
                        melhorou = True


            novos_offsprings.append((individuo, custo_individuo))

        # TODO: Pesquisar se o perf é o mais adequado
        tempo_fim = time.perf_counter()
        tempos.append(tempo_fim - tempo_começo)

        # print(f"Tamanho do offspring: {len(novos_offsprings)}")
        # print(novos_offsprings)
        print(f"Tamanho da pop: {len(populacao_atual)}")

        # NOTE : Prof falou para rever nosso elitimso!!
        populacao_atual = renovacao(populacao_atual, novos_offsprings, len(populacao_atual))

        melhor_custo_geracao = populacao_atual[0][1]
        melhores.append(populacao_atual[0])

        print(f"Geração {iteracao + 1}/{numero_iteracoes} | Melhor Custo: {melhor_custo_geracao:.2f} | Pop: {len(populacao_atual)}\n")

    return populacao_atual, melhores, tempos

### Wrappers pra simplificar a chamada aos métodos

In [36]:
def m_vmp(grafo):
    return vizinho_mais_proximo(grafo, 1)

def m_imb(grafo):
    return insercao_mais_barata(grafo, 1)

def m_vmp_bl(grafo):
    rota, custo = vizinho_mais_proximo(grafo, 1)
    for h in [swap, shift, inversao]:
        rota, custo = busca_local_primeira_melhoria(rota, custo, h, grafo_np)
    return rota, custo

def m_imb_bl(grafo):
    rota, custo = insercao_mais_barata(grafo, 1)
    for h in [swap, shift, inversao]:
        rota, custo = busca_local_primeira_melhoria(rota, custo, h, grafo_np)
    return rota, custo

# HACK: reconstruindo a população aqui pra simplificar. puxar pra uma outra célula dps
def m_ga(grafo, populacao):
    resultado = algoritmo_genetico(
        grafo, populacao, numero_iteracoes=100, taxa_mutacao=0.4,
        mutacao=mutacao_inversao, selecao=elitismo,
        cruzamento=cruzamento, renovacao=renovacao_elitismo,
    )
    return min(resultado, key=lambda x: x[1])

def m_memetico(grafo, populacao):
    pop_final, _, _ = algoritmo_memetico(
        grafo, populacao, numero_iteracoes=20, taxa_mutacao=0.3,
        mutacao=mutacao_inversao, selecao=elitismo,
        cruzamento=cruzamento, renovacao=renovacao_elitismo,
        heuristicas_busca_local=[swap, shift, inversao],
        probabilidade_busca_local=0.2,
    )
    return min(pop_final, key=lambda x: x[1])


In [37]:
# # Inicialização das submatrizes a partir dos DataFrames originais
# # 36 Cidades
# matriz_problema_km_36 = matriz_problema_km[list(range(1, 37))].iloc[0:36].copy()
# matriz_problema_min_36 = matriz_problema_min[list(range(1, 37))].iloc[0:36].copy()
#
# # 24 Cidades
# matriz_problema_km_24 = matriz_problema_km[list(range(1, 25))].iloc[0:24].copy()
# matriz_problema_min_24 = matriz_problema_min[list(range(1, 25))].iloc[0:24].copy()
#
# # 12 Cidades
# matriz_problema_km_12 = matriz_problema_km[list(range(1, 13))].iloc[0:12].copy()
# matriz_problema_min_12 = matriz_problema_min[list(range(1, 13))].iloc[0:12].copy()
#
# # 7 Cidades
# cidades_7 = [1, 7, 8, 9, 10, 11, 12]
# indices_7 = [c - 1 for c in cidades_7]
# matriz_problema_km_7 = matriz_problema_km[cidades_7].iloc[indices_7].copy()
# matriz_problema_km_7.index = range(len(cidades_7))
# matriz_problema_km_7.columns = range(1, len(cidades_7) + 1)
#
# matriz_problema_min_7 = matriz_problema_min[cidades_7].iloc[indices_7].copy()
# matriz_problema_min_7.index = range(len(cidades_7))
# matriz_problema_min_7.columns = range(1, len(cidades_7) + 1)
#
# # 6 Cidades
# matriz_problema_km_6 = matriz_problema_km[list(range(1, 7))].iloc[0:6].copy()
# matriz_problema_min_6 = matriz_problema_min[list(range(1, 7))].iloc[0:6].copy()
#
# # Definição das instâncias mapeadas
# instancias_problema = [
#     ('km_48', matriz_problema_km, 0),
#     ('min_48', matriz_problema_min, 1),
#     ('km_36', matriz_problema_km_36, 2),
#     ('min_36', matriz_problema_min_36, 3),
#     ('km_24', matriz_problema_km_24, 4),
#     ('min_24', matriz_problema_min_24, 5),
#     ('km_12', matriz_problema_km_12, 6),
#     ('min_12', matriz_problema_min_12, 7),
#     ('km_7', matriz_problema_km_7, 8),
#     ('min_7', matriz_problema_min_7, 9),
#     ('km_6', matriz_problema_km_6, 10),
#     ('min_6', matriz_problema_min_6, 11)
# ]
#
# print(f'Total de instâncias mapeadas: {len(instancias_problema)}')

In [38]:

SEEDS = list(range(1000, 1020))
N_SEEDS = len(SEEDS)

### Experimentação Comparativa (GA vs. Memético)


In [39]:
def executar_experimento_completo(instancias, seeds):
    resultados = {}
    for tag, matriz, idx in instancias:
        print(f"Executando {tag}...")
        resultados[tag] = {
            'GA': {'custos': [], 'tempos': [], 'melhor_rota': None, 'melhor_custo': float('inf')},
            'Memético': {'custos': [], 'tempos': [], 'melhor_rota': None, 'melhor_custo': float('inf')}
        }

        contador_genetico = 0
        contador_memetico = 0
        for seed in seeds:
            contador_genetico += 1
            random.seed(seed)
            np.random.seed(seed)
            populacao = criar_populacao(
                matriz, resultados_imb, resultados_vmp,
                resultados_imb_busca_local, resultados_vmp_busca_local,
                idx
            )
            t0 = time.perf_counter()
            melhor_ind = m_ga(matriz, populacao)
            tempo = time.perf_counter() - t0

            custo = melhor_ind[1]
            rota = melhor_ind[0]

            resultados[tag]['GA']['custos'].append(custo)
            resultados[tag]['GA']['tempos'].append(tempo)
            if custo < resultados[tag]['GA']['melhor_custo']:
                resultados[tag]['GA']['melhor_custo'] = custo
                resultados[tag]['GA']['melhor_rota'] = rota

            contador_memetico += 1
            print(f"Memético: iteração {contador_memetico}")
            t0 = time.perf_counter()
            melhor_ind = m_memetico(matriz, populacao)
            tempo = time.perf_counter() - t0

            custo = melhor_ind[1]
            rota = melhor_ind[0]

            resultados[tag]['Memético']['custos'].append(custo)
            resultados[tag]['Memético']['tempos'].append(tempo)
            if custo < resultados[tag]['Memético']['melhor_custo']:
                resultados[tag]['Memético']['melhor_custo'] = custo
                resultados[tag]['Memético']['melhor_rota'] = rota

    return resultados

In [40]:
resultados_finais = executar_experimento_completo(instancias_problema, SEEDS)

Streaming output truncated to the last 5000 lines.
Geração 14/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 15/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 16/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 17/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 18/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 19/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 20/20 | Melhor Custo: 438.30 | Pop: 40

População problema 9 criada. Tamanho final (sem duplicatas): 40 | Melhor custo: 438.30
Memético: iteração 2
Numero de genitores: 20
Tamanho da pop: 40
Geração 1/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 2/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 3/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 4/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 5/20 | Melhor Custo: 438.30 | Pop: 40

Tamanho da pop: 40
Geração 6/20 | Mel

### Análise Estatística dos Resultados
Consolidação dos dados em formato tabular exibindo média, desvio padrão, melhor resultado, tempo de execução e contagem de vitórias em formato de pareamento direto.

In [41]:
linhas_tabela = []
for tag, _, _ in instancias_problema:
    ga_custos = np.array(resultados_finais[tag]['GA']['custos'])
    mem_custos = np.array(resultados_finais[tag]['Memético']['custos'])

    ga_tempos = np.array(resultados_finais[tag]['GA']['tempos'])
    mem_tempos = np.array(resultados_finais[tag]['Memético']['tempos'])

    # Identifica o melhor valor absoluto encontrado para cada método
    ga_melhor_valor = ga_custos.min()
    mem_melhor_valor = mem_custos.min()

    # Calcula a frequência (número de vezes) que este melhor valor foi obtido nas 30 sementes
    # Usamos np.isclose ou tolerância mínima para evitar imprecisões de ponto flutuante
    ga_freq = np.sum(np.isclose(ga_custos, ga_melhor_valor))
    mem_freq = np.sum(np.isclose(mem_custos, mem_melhor_valor))

    # Contagem de pareamento direto (quantas vezes cada um foi estritamente melhor por semente)
    vitorias_ga = np.sum(ga_custos < mem_custos)
    vitorias_mem = np.sum(mem_custos < ga_custos)
    empates = np.sum(ga_custos == mem_custos)

    linhas_tabela.append({
        'Instância': tag,
        'GA Custo Médio': f"{ga_custos.mean():.2f} ± {ga_custos.std():.2f}",
        'Memético Custo Médio': f"{mem_custos.mean():.2f} ± {mem_custos.std():.2f}",
        'GA Melhor': f"{ga_melhor_valor:.2f}",
        'GA Freq. Melhor': f"{ga_freq}x",
        'Memético Melhor': f"{mem_melhor_valor:.2f}",
        'Memético Freq. Melhor': f"{mem_freq}x",
        'GA Tempo Médio (s)': f"{ga_tempos.mean():.3f}s",
        'Memético Tempo Médio (s)': f"{mem_tempos.mean():.3f}s",
        'Vitórias (GA/Mem/Empate)': f"{vitorias_ga} / {vitorias_mem} / {empates}"
    })

df_comparativo = pd.DataFrame(linhas_tabela)
display(df_comparativo)

,Instância,GA Custo Médio,Memético Custo Médio,GA Melhor,GA Freq. Melhor,Memético Melhor,Memético Freq. Melhor,GA Tempo Médio (s),Memético Tempo Médio (s),Vitórias (GA/Mem/Empate)
0,km_48,2046.00 ± 0.85,1947.71 ± 6.86,2042.30,1x,1942.30,10x,0.247s,46.653s,0 / 20 / 0
1,min_48,2059.60 ± 43.31,1976.70 ± 4.14,2006.00,2x,1973.00,9x,0.257s,44.174s,0 / 20 / 0
2,km_36,1695.00 ± 0.00,1695.00 ± 0.00,1695.00,20x,1695.00,20x,0.189s,15.062s,0 / 0 / 20
3,min_36,1724.15 ± 7.99,1662.55 ± 0.97,1701.00,1x,1662.00,13x,0.189s,14.257s,0 / 20 / 0
4,km_24,1321.00 ± 0.00,1321.00 ± 0.00,1321.00,20x,1321.00,20x,0.142s,3.626s,0 / 18 / 2
5,min_24,1228.00 ± 0.00,1223.40 ± 1.20,1228.00,20x,1223.00,18x,0.148s,3.452s,0 / 20 / 0
6,km_12,672.70 ± 0.00,672.70 ± 0.00,672.70,20x,672.70,20x,0.098s,0.380s,0 / 0 / 20
7,min_12,609.00 ± 0.00,606.00 ± 0.00,609.00,20x,606.00,20x,0.081s,0.305s,0 / 20 / 0
8,km_7,438.30 ± 0.00,438.30 ± 0.00,438.30,20x,438.30,20x,0.063s,0.059s,0 / 0 / 20
9,min_7,364.00 ± 0.00,364.00 ± 0.00,364.00,20x,364.00,20x,0.103s,0.101s,0 / 0 / 20


In [42]:

df_comparativo.to_csv('comparativo_ag_am_provisorio.csv')

In [43]:
resultados_finais_df = pd.DataFrame.from_dict(resultados_finais)
resultados_finais_df.to_csv("resultados_finais_ag_am_provisorio.csv")